## 1. Thư viện và thư mục đầu ra 

In [ ]:
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests
from IPython.display import display

ROOT = Path.cwd().resolve()
OUT = ROOT / "extra_data"
OUT.mkdir(parents=True, exist_ok=True)

## 2. Cấu hình API và khoảng ngày

In [ ]:
BASE_URL = "https://dulieu.dev/api"
START_DATE = date(2026, 1, 1)
END_DATE = date(2026, 11, 6)

API_LIMIT = 500
MAX_CALLS_PER_RUN = 80
CHUNK_DAYS = {
    "coffee": 31,
    "exchange": 30,
    "commodities": 14,
    "fuel": 14,
    "weather": 60,
}

REGION_CITY = {
    "Gia Lai": "Pleiku",
    "Lâm Đồng": "Đà Lạt",
    "Đắk Lắk": "Buôn Ma Thuột",
    "Đắk Nông": "Gia Nghĩa",
    "Bình Phước": "Đồng Xoài",
    "Bà Rịa - Vũng Tàu": "Bà Rịa",
}
SYMBOLS = {"ROBUSTA", "WTI", "BRENT"}

FILES = {
    "coffee": OUT / "coffee_domestic_raw.csv",
    "exchange": OUT / "exchange_usd_history.csv",
    "commodities": OUT / "commodities_api_raw_v2.csv",
    "commodities_selected": OUT / "commodities_history_v2.csv",
    "fuel": OUT / "fuel_history_new.csv",
    "weather": OUT / "weather_history_new.csv",
}

API_KEY = getpass("Nhập API key DuLieu.dev (ẩn ký tự): ").strip()
if not API_KEY:
    raise ValueError("Chưa nhập API key")

## 3. Hàm hỗ trợ chung

In [ ]:
class CrawlPaused(Exception):
    """Tạm dừng khi hết lượt gọi hoặc phản hồi API không đạt yêu cầu."""


def date_ranges(start, end, chunk_days):
    current = start
    while current <= end:
        last = min(end, current + timedelta(days=chunk_days - 1))
        yield current, last
        current = last + timedelta(days=1)


def make_job(start, end, params=None, metadata=None):
    return {
        "start": start.isoformat(),
        "end": end.isoformat(),
        "params": params or {},
        "metadata": metadata or {},
    }


def create_jobs(endpoint):
    days = CHUNK_DAYS[endpoint]
    if endpoint != "weather":
        return [make_job(a, b) for a, b in date_ranges(START_DATE, END_DATE, days)]
    jobs = []
    for region, city in REGION_CITY.items():
        for a, b in date_ranges(START_DATE, END_DATE, days):
            jobs.append(make_job(a, b, {"city": city},
                                 {"target_region": region, "requested_city": city}))
    return jobs


def atomic_json(path, data):
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(temp, path)


def atomic_csv(path, frame):
    temp = path.with_name(path.name + ".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8-sig")
    os.replace(temp, path)


def append_csv(path, rows, metadata=None):
    if not rows:
        return
    batch = pd.json_normalize(rows, sep="_").fillna("").astype(str)
    for key, value in (metadata or {}).items():
        batch[key] = str(value)
    if path.exists():
        previous = pd.read_csv(path, dtype=str, keep_default_na=False)
        batch = pd.concat([previous, batch], ignore_index=True)
    # Loại trùng hoàn toàn để việc chạy lại sau gián đoạn không nhân đôi bản ghi.
    batch = batch.drop_duplicates(ignore_index=True)
    atomic_csv(path, batch)


class APIClient:
    def __init__(self, key):
        self.session = requests.Session()
        self.session.headers.update({"Authorization": f"Bearer {key}"})
        self.calls = 0
        self.remaining = None

    def fetch(self, endpoint, job):
        if self.calls >= MAX_CALLS_PER_RUN or (
            self.remaining is not None and self.remaining <= 1
        ):
            raise CrawlPaused("Đã chạm ngưỡng lượt API; chạy tiếp khi có quota")

        params = {"from": job["start"], "to": job["end"], "limit": API_LIMIT, **job["params"]}
        self.calls += 1
        try:
            response = self.session.get(f"{BASE_URL}/{endpoint}", params=params, timeout=40)
        except requests.RequestException as exc:
            raise CrawlPaused(f"/{endpoint}: lỗi kết nối: {exc}") from exc

        header = response.headers.get("X-RateLimit-Remaining")
        if header is not None:
            try:
                self.remaining = int(header)
            except ValueError:
                pass
        if response.status_code in (401, 403, 429):
            raise CrawlPaused(f"/{endpoint}: HTTP {response.status_code} (key/quota/quyền)")
        try:
            response.raise_for_status()
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            raise CrawlPaused(f"/{endpoint}: phản hồi không hợp lệ: {exc}") from exc

        rows = payload.get("data") if isinstance(payload, dict) else None
        if not isinstance(rows, list) or not all(isinstance(row, dict) for row in rows):
            raise CrawlPaused(f"/{endpoint}: không có mảng data hợp lệ")

        first = date.fromisoformat(job["start"])
        last = date.fromisoformat(job["end"])
        for row in rows:
            raw_date = str(row.get("date", ""))[:10]
            try:
                observed = date.fromisoformat(raw_date)
            except ValueError as exc:
                raise CrawlPaused(f"/{endpoint}: date không hợp lệ: {raw_date!r}") from exc
            if not first <= observed <= last:
                raise CrawlPaused(f"/{endpoint}: API trả ngày {raw_date} ngoài {first} - {last}")

        print(f"/{endpoint:11s} {job['start']} → {job['end']} | "
              f"{len(rows):4d} dòng | calls={self.calls} | quota={self.remaining}")
        time.sleep(0.2)
        return rows


client = APIClient(API_KEY)
del API_KEY 

## 4. Thu thập và lưu checkpoint theo nguồn

In [ ]:
def save_commodities_subset():
    raw_path = FILES["commodities"]
    if not raw_path.exists():
        return
    frame = pd.read_csv(raw_path, dtype=str, keep_default_na=False)
    if "symbol" not in frame.columns:
        print("CẢNH BÁO: Commodities không có cột symbol; chỉ lưu CSV thô")
        return
    chosen = frame.loc[frame["symbol"].str.strip().str.upper().isin(SYMBOLS)].copy()
    atomic_csv(FILES["commodities_selected"], chosen)


def select_rows(endpoint, rows):
    if endpoint == "exchange":
        return [row for row in rows if str(row.get("currency", "")).strip().upper() == "USD"]
    return rows


def checkpoint_config(endpoint):
    """Cấu hình gắn với tiến độ, để không bỏ sót ngày nếu người dùng đổi phạm vi."""
    return {
        "endpoint": endpoint,
        "start": START_DATE.isoformat(),
        "end": END_DATE.isoformat(),
        "chunk_days": CHUNK_DAYS[endpoint],
        "cities": REGION_CITY if endpoint == "weather" else {},
    }


def collect(endpoint):
    if endpoint not in CHUNK_DAYS:
        raise ValueError(f"Nguồn không hợp lệ: {endpoint}")
    output = FILES[endpoint]
    checkpoint = OUT / f"{endpoint}_progress.json"
    config = checkpoint_config(endpoint)

    if checkpoint.exists():
        state = json.loads(checkpoint.read_text(encoding="utf-8"))
        if state.get("config") != config:
            raise RuntimeError(
                f"Checkpoint {checkpoint.name} không khớp cấu hình hiện tại. "
                "Hãy sao lưu CSV và checkpoint cũ rồi tạo thư mục dữ liệu mới."
            )
        pending, done = state["pending"], state.get("done", [])
        if any(job.get("saved_rows", 0) > 0 for job in done) and not output.exists():
            raise RuntimeError(f"Checkpoint báo đã lưu dữ liệu nhưng thiếu {output.name}")
    else:
        if output.exists():
            raise RuntimeError(
                f"Có {output.name} nhưng không có checkpoint tương ứng. "
                "Hãy chuyển các file cũ ra khỏi extra_data trước khi chạy."
            )
        pending, done = create_jobs(endpoint), []
        atomic_json(checkpoint, {"config": config, "pending": pending, "done": done})

    def save_progress():
        atomic_json(checkpoint, {"config": config, "pending": pending, "done": done})

    print(f"\n=== {endpoint.upper()} | đã xong: {len(done)} | còn: {len(pending)} ===")
    while pending:
        job = pending[0]
        try:
            rows = client.fetch(endpoint, job)
        except CrawlPaused as exc:
            print("TẠM DỪNG:", exc)
            break

        if len(rows) >= API_LIMIT:
            first, last = date.fromisoformat(job["start"]), date.fromisoformat(job["end"])
            if first == last:
                print("TẠM DỪNG: một ngày đạt 500 dòng; cần API phân trang để thu thập đủ")
                break
            midpoint = first + timedelta(days=(last - first).days // 2)
            pending[:1] = [
                make_job(first, midpoint, job["params"], job["metadata"]),
                make_job(midpoint + timedelta(days=1), last, job["params"], job["metadata"]),
            ]
            save_progress()
            print("  Chia đôi khoảng ngày do chạm API_LIMIT")
            continue

        selected = select_rows(endpoint, rows)
        extra_columns = dict(job["metadata"])
        if endpoint == "coffee":
            extra_columns["commodity"] = "coffee"
        append_csv(output, selected, extra_columns)

        if endpoint == "commodities":
            save_commodities_subset()
        if endpoint == "weather" and rows:
            actual_cities = sorted({str(row.get("city", "")) for row in rows})
            print("  City thực tế API trả về:", ", ".join(actual_cities))

        pending.pop(0)
        done.append({**job, "saved_rows": len(selected)})
        save_progress()

    if endpoint == "commodities" and not pending:
        save_commodities_subset()
    print(f"{endpoint}: hoàn tất {len(done)} khoảng | còn {len(pending)} "
          f"| tổng calls trong kernel: {client.calls}")

## 5. Thu thập Coffee — giá cà phê trong nước

In [ ]:
collect("coffee")

## 6. Thu thập Exchange — tỷ giá USD

In [ ]:
collect("exchange")

## 7. Thu thập Commodities — Robusta, WTI, Brent

In [ ]:
collect("commodities")

## 8. Thu thập Fuel — giá nhiên liệu trong nước

In [ ]:
collect("fuel")

## 9. Thu thập Weather — thời tiết theo khu vực

In [ ]:
collect("weather")

## 10. Kiểm tra dữ liệu CSV và tiến độ (không gọi API) 

In [ ]:
summary = []
for name, path in FILES.items():
    if not path.exists():
        summary.append({"source": name, "file": path.name, "rows": 0,
                        "columns": 0, "first_date": "", "last_date": ""})
        continue

    frame = pd.read_csv(path, low_memory=False)
    dates = (pd.to_datetime(frame["date"], errors="coerce").dropna()
             if "date" in frame.columns else pd.Series(dtype="datetime64[ns]"))
    summary.append({
        "source": name,
        "file": path.name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "first_date": str(dates.min().date()) if not dates.empty else "",
        "last_date": str(dates.max().date()) if not dates.empty else "",
    })
display(pd.DataFrame(summary))

for endpoint in CHUNK_DAYS:
    checkpoint = OUT / f"{endpoint}_progress.json"
    if checkpoint.exists():
        progress = json.loads(checkpoint.read_text(encoding="utf-8"))
        print(f"{endpoint:12s} đã xong: {len(progress['done']):3d} "
              f"| còn: {len(progress['pending']):3d}")
print("Số lần thử gọi API trong kernel:", client.calls)
print("Thư mục CSV:", OUT)